# Part 2: Customer Spending Data

In [2]:
# import the Custormer Spending  dataset
import pandas as pd
data_raw = pd.read_csv(r'customer_spending.csv')

# saving raw data as new dataframe for exploration
data = data_raw.copy()
data.head()

,sale_date,sale_year,sale_month,age,gender,country,state,category,sub_category,quantity,unit_cost,unit_price,cost,revenue
0,2016-02-19,2016,February,29,F,United States,Washington,Accessories,Tires and Tubes,1,80.00,109.000000,80,109
1,2016-02-20,2016,February,29,F,United States,Washington,Clothing,Gloves,2,24.50,28.500000,49,57
2,2016-02-27,2016,February,29,F,United States,Washington,Accessories,Tires and Tubes,3,3.67,5.000000,11,15
3,2016-03-12,2016,March,29,F,United States,Washington,Accessories,Tires and Tubes,2,87.50,116.500000,175,233
4,2016-03-12,2016,March,29,F,United States,Washington,Accessories,Tires and Tubes,3,35.00,41.666667,105,125


In [3]:
# To check the Data information
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 34866 entries, 0 to 34865
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sale_date     34866 non-null  str    
 1   sale_year     34866 non-null  int64  
 2   sale_month    34866 non-null  str    
 3   age           34866 non-null  int64  
 4   gender        34866 non-null  str    
 5   country       34866 non-null  str    
 6   state         34866 non-null  str    
 7   category      34866 non-null  str    
 8   sub_category  34866 non-null  str    
 9   quantity      34866 non-null  int64  
 10  unit_cost     34866 non-null  float64
 11  unit_price    34866 non-null  float64
 12  cost          34866 non-null  int64  
 13  revenue       34866 non-null  int64  
dtypes: float64(2), int64(5), str(7)
memory usage: 3.7 MB


In [4]:
data.describe()

,sale_year,age,quantity,unit_cost,unit_price,cost,revenue
count,34866.000000,34866.000000,34866.000000,34866.000000,34866.000000,34866.000000,34866.000000
mean,2015.569237,36.382895,2.002524,349.880567,389.232485,576.004532,640.870074
std,0.495190,11.112902,0.813936,490.015846,525.319091,690.500395,736.650597
min,2015.000000,17.000000,1.000000,0.670000,0.666667,2.000000,2.000000
25%,2015.000000,28.000000,1.000000,45.000000,53.666667,85.000000,102.000000
50%,2016.000000,35.000000,2.000000,150.000000,179.000000,261.000000,319.000000
75%,2016.000000,44.000000,3.000000,455.000000,521.000000,769.000000,902.000000
max,2016.000000,87.000000,3.000000,3240.000000,5082.000000,3600.000000,5082.000000


In [5]:
# To check for duplicate in the dataset
data.duplicated().sum()

np.int64(1)

Write a query that returns each category and the corresponding total revenue for that 
category for the sale_year 2016. The aggregated column should be named 
total_revenue. The output should be arranged alphabetically. 

In [6]:
# Total revenue by category for sale_year 2016, sorted alphabetically.
revenue_by_category = (
    data[data['sale_year'] == 2016]
    .groupby('category', as_index=False)['revenue'].sum()
    .rename(columns={'revenue': 'total_revenue'})
    .sort_values('category')
)
revenue_by_category

,category,total_revenue
0,Accessories,4594897
1,Bikes,5722257
2,Clothing,2079651


Write a query that returns a list of sub_categories and their corresponding average 
unit_price (named avg_unit_price), average unit_cost named avg_unit_cost, as well as 
the difference between these two values (named margin) for the sale_year 2015. Round 
all values to two decimal places. Organize the results alphabetically. 

In [7]:
# Average prices, costs, and margins by sub_category for sale_year 2015.
sub_category = (data[data['sale_year'] == 2015]
    .groupby('sub_category', as_index=False)
    .agg(
        avg_unit_price=('unit_price', 'mean'),
        avg_unit_cost=('unit_cost', 'mean')
    )
)
sub_category['margin'] = (
    sub_category['avg_unit_price'] - sub_category['avg_unit_cost']
)
sub_category = (
    sub_category
    .round({'avg_unit_price': 2, 'avg_unit_cost': 2, 'margin': 2})
    .sort_values('sub_category')
)
sub_category

,sub_category,avg_unit_price,avg_unit_cost,margin
0,Bike Stands,602.00,528.61,73.40
1,Bottles and Cages,75.82,66.88,8.94
2,Caps,99.96,90.38,9.58
3,Cleaners,94.85,84.20,10.65
4,Helmets,368.32,322.19,46.13
5,Hydration Packs,545.69,481.44,64.25
6,Jerseys,511.79,462.61,49.18
7,Mountain Bikes,1108.03,1147.46,-39.43
8,Road Bikes,799.99,818.16,-18.17
9,Shorts,728.04,679.28,48.76


Write a query that returns the sale_year and corresponding total number of female 
buyers (gender) for each sale_year who made purchases in the Clothing category. 
Name the aggregated column total_female_buyers.

In [8]:
# Total female buyers in the Clothing category by sale_year.
female_clothing_buyers = (
    data[(data['category'] == 'Clothing') & (data['gender'] == 'F')]
    .groupby('sale_year')
    .size()
    .reset_index(name='total_female_buyers')
    .sort_values('sale_year')
)
female_clothing_buyers

,sale_year,total_female_buyers
0,2015,1037
1,2016,1468


Write a query that returns the age, sub_cateogry, average quantity (as a whole 
number, named avg_quantity), and average cost of products (rounded to 2 decimals, 
named avg_cost) purchased by each age and sub_category. Organize the data by age, 
oldest to youngest, and then by sub_category alphabetically.

In [9]:
# Average quantity and cost by age and sub_category, sorted by age descending.
age_sub_category_summary = (
    data
    .groupby(['age', 'sub_category'], as_index=False)
    .agg(
        avg_quantity=('quantity', 'mean'),
        avg_cost=('cost', 'mean')
    )
)

age_sub_category_summary['avg_quantity'] = (
    age_sub_category_summary['avg_quantity'].round().astype(int)
)

age_sub_category_summary['avg_cost'] = (
    age_sub_category_summary['avg_cost'].round(2)
)

age_sub_category_summary = age_sub_category_summary.sort_values(
    ['age', 'sub_category'],
    ascending=[False, True]
)

age_sub_category_summary

,age,sub_category,avg_quantity,avg_cost
889,87,Bike Racks,3,240.00
890,87,Tires and Tubes,2,44.50
886,86,Fenders,3,637.00
887,86,Tires and Tubes,2,166.50
888,86,Vests,1,1842.00
...,...,...,...,...
11,17,Shorts,2,787.50
12,17,Socks,2,171.00
13,17,Tires and Tubes,2,204.34
14,17,Touring Bikes,2,1278.71


 query that returns a list of countries where more than 900 transactions were 
made by customers between the ages of 18-25 (inclusive)

In [10]:
# Countries with more than 900 transactions by customers aged 18 through 25.
country_transaction_counts = (
    data[data['age'].between(18, 25)]
    .groupby('country')
    .size()
    .reset_index(name='transaction_count')
)
countries_over_900 = country_transaction_counts[
    country_transaction_counts['transaction_count'] > 900
]
countries_over_900

,country,transaction_count
1,Germany,976
2,United Kingdom,1101
3,United States,2823


Write a query to identify which sale_year and category combinations were the most 
profitable. Return the columns sale_year, category, the total revenue as total_revenue, 
the total cost as total_cost, and the difference between the two as profit for each 
sale_year and category. Include only rows that were profitable and sort the results from 
highest profit to lowest.

In [11]:
# Profitable sale_year and category combinations, sorted by profit descending.
profitability = (
    data
    .groupby(['sale_year', 'category'], as_index=False)
    .agg(
        total_revenue=('revenue', 'sum'),
        total_cost=('cost', 'sum')
    )
)
profitability['profit'] = profitability['total_revenue'] - profitability['total_cost']
profitable_combinations = (
    profitability[profitability['profit'] > 0]
    .sort_values('profit', ascending=False)
)
profitable_combinations

,sale_year,category,total_revenue,total_cost,profit
3,2016,Accessories,4594897,3559646,1035251
4,2016,Bikes,5722257,5209322,512935
5,2016,Clothing,2079651,1654855,424796
0,2015,Accessories,2825767,2482249,343518
2,2015,Clothing,1357906,1237470,120436


Write a query that calculates the average spending per age for male customers 
(gender), where spending is based on the product of unit_price and quantity. Return 
the age and the calculated average spending, rounded to two decimal places, as 
avg_spending. Organize the results from highest to lowest avg_spending.

In [12]:
# Average spending per age for male customers, sorted by average spending descending.
male_spending = data[data['gender'] == 'M'].copy()
male_spending['spending'] = male_spending['unit_price'] * male_spending['quantity']
average_spending_by_age = (
    male_spending
    .groupby('age', as_index=False)['spending']
    .mean()
    .rename(columns={'spending': 'avg_spending'})
)
average_spending_by_age['avg_spending'] = average_spending_by_age['avg_spending'].round(2)
average_spending_by_age = average_spending_by_age.sort_values(
    'avg_spending', ascending=False
)
average_spending_by_age

,age,avg_spending
65,86,2026.00
56,73,1170.00
57,74,971.00
58,75,869.00
55,72,764.00
...,...,...
60,78,257.75
61,79,182.60
59,76,159.00
62,81,93.00


Write a query to determine the highest unit_cost, lowest unit_cost, and average 
unit_cost for each gender in each category. The output columns should be gender, 
category, high_cost, low_cost, avg_cost, in that order. Organize the results by gender 
and then category.

In [14]:
# Unit cost statistics for each gender and category.
cost_summary = (
    data
    .groupby(['gender', 'category'], as_index=False)
    .agg(
        high_cost=('unit_cost', 'max'),
        low_cost=('unit_cost', 'min'),
        avg_cost=('unit_cost', 'mean')
    )
    .sort_values(['gender', 'category'])
)
cost_summary

,gender,category,high_cost,low_cost,avg_cost
0,F,Accessories,3120.0,0.67,159.301847
1,F,Bikes,2443.0,180.00,956.653830
2,F,Clothing,2100.0,3.00,334.904970
3,M,Accessories,3240.0,0.67,166.962106
4,M,Bikes,2443.0,180.00,949.309243
5,M,Clothing,2100.0,3.00,337.600596


Write a query to determine the age distribution of customers by category and country 
for the sale_year 2016. For each category and country, calculate the age of the 
youngest and oldest customers, and the average age of customers rounded to one 
decimal place. The output columns should be category, country, youngest_customer, 
oldest_customer, and avg_customer_age, in that order. Organize your results by 
category and then by the average age of customers.

In [15]:
# Customer age distribution by category and country for sale_year 2016.
age_distribution = (
    data[data['sale_year'] == 2016]
    .groupby(['category', 'country'], as_index=False)
    .agg(
        youngest_customer=('age', 'min'),
        oldest_customer=('age', 'max'),
        avg_customer_age=('age', 'mean')
    )
)
age_distribution['avg_customer_age'] = age_distribution['avg_customer_age'].round(1)
age_distribution = (
    age_distribution[
        ['category', 'country', 'youngest_customer', 'oldest_customer', 'avg_customer_age']
    ]
    .sort_values(['category', 'avg_customer_age'])
)
age_distribution

,category,country,youngest_customer,oldest_customer,avg_customer_age
1,Accessories,Germany,17,87,35.4
0,Accessories,France,17,84,35.6
2,Accessories,United Kingdom,17,85,36.1
3,Accessories,United States,17,78,37.8
4,Bikes,France,17,59,34.0
5,Bikes,Germany,17,72,34.6
6,Bikes,United Kingdom,17,75,34.8
7,Bikes,United States,17,72,39.7
9,Clothing,Germany,17,86,34.2
8,Clothing,France,17,84,35.3


Write a query to return the country that has the highest average revenue (rounded to 2 
decimals). Your output columns should be country and high_sales.

In [16]:
# Country with the highest average revenue.
country_revenue = (
    data.groupby('country', as_index=False)['revenue']
    .mean()
    .rename(columns={'revenue': 'high_sales'})
)
country_revenue['high_sales'] = country_revenue['high_sales'].round(2)
country_revenue = country_revenue.sort_values('high_sales', ascending=False)
country_revenue.head(1)

,country,high_sales
1,Germany,816.09
